# Pattern Feature Engineering

8 raw JSON sources → 51 features per (stock, date) → Parquet output

**Storage**: All data on Google Drive (`MyDrive/Stock/data/pattern_data/`)
- Raw data: `raw/` (~4.3GB: institutional, financials, broker, margin, tdcc, revenue, news, industry)
- Output: `features/` (~360MB: features_all, forward_returns, price_cache, winner_dna)

**Dimensions** (51 features):
- Technical (20): price/volume derived
- Institutional (15): 三大法人 + margin + TDCC + broker
- Industry (5): sector RS, peer alpha, chain position
- Fundamental (8): EPS, ROE, revenue growth, PE, PB
- News (3): count, sentiment, spike

**Output**:
- `features_all.parquet` — (stock_code, date, f1..f51)
- `forward_returns.parquet` — (stock_code, date, d3, d7, d21, d90, d180)
- `price_cache.parquet` — OHLCV for all stocks (used by R95.2 accumulation backtest)
- `feature_metadata.json` — dimension → feature column mapping

In [ ]:
# === Setup ===
import os, json, glob, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

# === Google Drive Mount ===
from google.colab import drive
drive.mount('/content/drive')

# === Config ===
# All data stored on Google Drive for persistence
DRIVE_ROOT = Path('/content/drive/MyDrive/Stock/data/pattern_data')
DATA_ROOT = DRIVE_ROOT / 'raw'
OUTPUT_DIR = DRIVE_ROOT / 'features'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Need yfinance for price data (OHLCV)
try:
    import yfinance as yf
except ImportError:
    !pip install yfinance -q
    import yfinance as yf

print(f'Data root: {DATA_ROOT}')
print(f'Output dir: {OUTPUT_DIR}')
for src in ['broker', 'institutional', 'financials', 'margin', 'tdcc', 'revenue', 'news', 'industry']:
    p = DATA_ROOT / src
    n = len(list(p.glob('*.json'))) if p.exists() else 0
    print(f'  {src}: {n} files')

## 1. Discover Stock Universe

Find all stock codes present in institutional data (most comprehensive daily source).

In [ ]:
import re

def discover_stock_codes():
    """Extract all stock codes from institutional daily files."""
    codes = set()
    inst_dir = DATA_ROOT / 'institutional'
    # Sample first 5 TWSE + 5 TPEX files
    twse_files = sorted(inst_dir.glob('twse_*.json'))[:5]
    tpex_files = sorted(inst_dir.glob('tpex_*.json'))[:5]
    
    for f in twse_files + tpex_files:
        with open(f, 'r', encoding='utf-8') as fp:
            d = json.load(fp)
        for row in d.get('data', []):
            code = row[0].strip()
            # Only 4-digit stock codes (skip ETFs, bonds, warrants)
            if re.match(r'^\d{4}$', code):
                codes.add(code)
    return sorted(codes)

all_codes = discover_stock_codes()
print(f'Discovered {len(all_codes)} stock codes')
print(f'Sample: {all_codes[:10]}')

## 2. Fetch Price Data (OHLCV)

Use yfinance to get daily OHLCV for all stocks (2020-01-01 to present).

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

PRICE_CACHE_FILE = OUTPUT_DIR / 'price_cache.parquet'
START_DATE = '2020-01-01'

def fetch_price_single(code: str) -> pd.DataFrame | None:
    """Fetch OHLCV for a single stock."""
    try:
        ticker = f'{code}.TW'
        df = yf.download(ticker, start=START_DATE, auto_adjust=True, progress=False)
        if df.empty:
            # Try .TWO for OTC
            ticker = f'{code}.TWO'
            df = yf.download(ticker, start=START_DATE, auto_adjust=True, progress=False)
        if df.empty:
            return None
        # yfinance columns are lowercase with auto_adjust=True
        df.columns = [c.lower() if isinstance(c, str) else c[0].lower() for c in df.columns]
        df = df[['open', 'high', 'low', 'close', 'volume']].copy()
        df['stock_code'] = code
        df.index.name = 'date'
        return df.reset_index()
    except Exception:
        return None

def fetch_all_prices(codes: list[str], max_workers: int = 8) -> pd.DataFrame:
    """Fetch prices for all stocks with threading."""
    if PRICE_CACHE_FILE.exists():
        print(f'Loading cached prices from {PRICE_CACHE_FILE}')
        return pd.read_parquet(PRICE_CACHE_FILE)
    
    results = []
    failed = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_price_single, c): c for c in codes}
        for i, future in enumerate(as_completed(futures)):
            code = futures[future]
            df = future.result()
            if df is not None and len(df) > 100:  # Minimum 100 days
                results.append(df)
            else:
                failed.append(code)
            if (i + 1) % 100 == 0:
                print(f'  Fetched {i+1}/{len(codes)} ({len(results)} success, {len(failed)} failed)')
    
    prices = pd.concat(results, ignore_index=True)
    prices['date'] = pd.to_datetime(prices['date'])
    prices.to_parquet(PRICE_CACHE_FILE, index=False)
    print(f'Saved price cache: {len(results)} stocks, {len(prices)} rows')
    print(f'Failed: {len(failed)} stocks')
    return prices

prices_df = fetch_all_prices(all_codes)
print(f'Price data shape: {prices_df.shape}')
print(f'Date range: {prices_df["date"].min()} to {prices_df["date"].max()}')
print(f'Stocks: {prices_df["stock_code"].nunique()}')

## 3. Technical Features (20)

All derived from OHLCV price data.

In [ ]:
def compute_technical_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute 20 technical features for a single stock's OHLCV data.
    
    Input: DataFrame with columns [date, open, high, low, close, volume]
    Output: DataFrame with 20 feature columns added
    """
    df = df.sort_values('date').copy()
    c = df['close']
    h = df['high']
    l = df['low']
    o = df['open']
    v = df['volume'].astype(float)
    
    # Returns
    df['ret_1d'] = c.pct_change(1)
    df['ret_5d'] = c.pct_change(5)
    df['ret_20d'] = c.pct_change(20)
    
    # Moving average ratios
    df['ma5_ratio'] = c / c.rolling(5).mean()
    df['ma20_ratio'] = c / c.rolling(20).mean()
    df['ma60_ratio'] = c / c.rolling(60).mean()
    
    # Bollinger Band position
    bb_mid = c.rolling(20).mean()
    bb_std = c.rolling(20).std()
    bb_upper = bb_mid + 2 * bb_std
    bb_lower = bb_mid - 2 * bb_std
    df['bb_position'] = (c - bb_lower) / (bb_upper - bb_lower)
    
    # RSI(14)
    delta = c.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(span=14, adjust=False).mean()
    avg_loss = loss.ewm(span=14, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df['rsi_14'] = (100 - 100 / (1 + rs)) / 100  # Normalize to 0-1
    
    # MACD histogram (normalized by price)
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    df['macd_hist'] = (macd_line - signal_line) / c  # Normalize by price
    
    # Stochastic K/D
    low_14 = l.rolling(14).min()
    high_14 = h.rolling(14).max()
    df['kd_k'] = ((c - low_14) / (high_14 - low_14).replace(0, np.nan)) 
    df['kd_d'] = df['kd_k'].rolling(3).mean()
    
    # ATR percentage
    tr = pd.concat([
        h - l,
        (h - c.shift(1)).abs(),
        (l - c.shift(1)).abs()
    ], axis=1).max(axis=1)
    df['atr_pct'] = tr.rolling(14).mean() / c
    
    # Volume ratios
    df['vol_ratio_5'] = v / v.rolling(5).mean().replace(0, np.nan)
    df['vol_ratio_20'] = v / v.rolling(20).mean().replace(0, np.nan)
    
    # Candle features
    hl_range = h - l
    df['high_low_range'] = hl_range / c
    df['close_vs_high'] = (c - l) / hl_range.replace(0, np.nan)
    
    # Gap
    df['gap_pct'] = (o - c.shift(1)) / c.shift(1)
    
    # Trend slope (linear regression slope over 20d, normalized)
    def rolling_slope(s, w=20):
        x = np.arange(w, dtype=float)
        x -= x.mean()
        result = s.rolling(w).apply(
            lambda y: np.polyfit(x, y, 1)[0] if len(y) == w else np.nan,
            raw=True
        )
        return result
    df['trend_slope_20'] = rolling_slope(c, 20) / c  # Normalize by price
    
    # Volatility (std of returns over 20d)
    df['volatility_20'] = df['ret_1d'].rolling(20).std()
    
    # RS Rating vs TAIEX (placeholder — will be computed separately)
    df['rs_rating'] = np.nan  # Filled later with market-relative strength
    
    return df

# Test on one stock
test_stock = prices_df[prices_df['stock_code'] == '2330'].copy()
test_tech = compute_technical_features(test_stock)
tech_cols = ['ret_1d', 'ret_5d', 'ret_20d', 'ma5_ratio', 'ma20_ratio', 'ma60_ratio',
             'bb_position', 'rsi_14', 'macd_hist', 'kd_k', 'kd_d', 'atr_pct',
             'vol_ratio_5', 'vol_ratio_20', 'high_low_range', 'close_vs_high',
             'gap_pct', 'trend_slope_20', 'volatility_20', 'rs_rating']
print(f'Technical features computed: {len(tech_cols)}')
print(test_tech[tech_cols].tail(3))

## 4. Institutional Features (15)

From institutional (三大法人), margin, TDCC, and broker data.

In [ ]:
def parse_number(s: str) -> float:
    """Parse number string with commas and signs."""
    if not s or s == '--' or s == '-':
        return 0.0
    return float(str(s).replace(',', '').strip())


def load_institutional_data() -> pd.DataFrame:
    """Parse TWSE + TPEX institutional daily files → per-stock daily DataFrame.
    
    Returns: DataFrame with columns:
      [date, stock_code, foreign_net, trust_net, dealer_net, total_net]
    """
    inst_dir = DATA_ROOT / 'institutional'
    rows = []
    
    for f in sorted(inst_dir.glob('*.json')):
        fname = f.stem  # e.g. twse_20200102 or tpex_20200102
        parts = fname.split('_')
        if len(parts) != 2:
            continue
        market, date_str = parts
        date = pd.Timestamp(f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}')
        
        with open(f, 'r', encoding='utf-8') as fp:
            d = json.load(fp)
        
        for row in d.get('data', []):
            code = row[0].strip()
            if not re.match(r'^\d{4}$', code):
                continue
            
            if market == 'twse':
                # TWSE fields: 0=code, 1=name, 2=外資買, 3=外資賣, 4=外資超,
                # 5=外資自營買, 6=外資自營賣, 7=外資自營超,
                # 8=投信買, 9=投信賣, 10=投信超,
                # 11=自營商超, ..., 18=三大法人合計
                foreign_net = parse_number(row[4]) + parse_number(row[7])  # 外資+外資自營
                trust_net = parse_number(row[10])
                dealer_net = parse_number(row[11])
                total_net = parse_number(row[18])
            else:  # tpex
                # TPEX fields: 0=code, 1=name,
                # 2=外資買, 3=外資賣, 4=外資超, 5=外資自營買, 6=外資自營賣, 7=外資自營超,
                # 8=投信買, 9=投信賣, 10=投信超, ..., 22=三大法人合計
                foreign_net = parse_number(row[4]) + parse_number(row[7]) if len(row) > 7 else 0
                trust_net = parse_number(row[10]) if len(row) > 10 else 0
                dealer_net = parse_number(row[11]) if len(row) > 11 else 0
                total_net = parse_number(row[-1]) if len(row) > 18 else 0
            
            rows.append({
                'date': date, 'stock_code': code,
                'foreign_net': foreign_net, 'trust_net': trust_net,
                'dealer_net': dealer_net, 'total_net': total_net
            })
    
    df = pd.DataFrame(rows)
    print(f'Institutional data: {len(df)} rows, {df["stock_code"].nunique()} stocks')
    return df

inst_df = load_institutional_data()
inst_df.head()

In [ ]:
def load_margin_data() -> pd.DataFrame:
    """Parse margin daily files → per-stock daily DataFrame.
    
    Returns: DataFrame with columns:
      [date, stock_code, margin_balance, short_balance, margin_utilization]
    """
    margin_dir = DATA_ROOT / 'margin'
    rows = []
    
    for f in sorted(margin_dir.glob('*.json')):
        fname = f.stem  # e.g. tpex_20200102
        parts = fname.split('_')
        if len(parts) != 2:
            continue
        market, date_str = parts
        date = pd.Timestamp(f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}')
        
        with open(f, 'r', encoding='utf-8') as fp:
            d = json.load(fp)
        
        for row in d.get('data', []):
            code = row[0].strip()
            if not re.match(r'^\d{4}$', code):
                continue
            # Fields: 0=code, 1=name, 2=前資餘額, 3=資買, 4=資賣, 5=現償, 6=資餘額,
            #         7=資屬證金, 8=資使用率(%), 9=資限額,
            #         10=前券餘額, 11=券賣, 12=券買, 13=券償, 14=券餘額, ...
            margin_balance = parse_number(row[6]) if len(row) > 6 else 0
            short_balance = parse_number(row[14]) if len(row) > 14 else 0
            margin_util = parse_number(row[8]) if len(row) > 8 else 0
            
            rows.append({
                'date': date, 'stock_code': code,
                'margin_balance': margin_balance,
                'short_balance': short_balance,
                'margin_utilization': margin_util / 100.0  # Convert to 0-1
            })
    
    df = pd.DataFrame(rows)
    print(f'Margin data: {len(df)} rows, {df["stock_code"].nunique()} stocks')
    return df

margin_df = load_margin_data()
margin_df.head()

In [ ]:
def load_tdcc_data() -> pd.DataFrame:
    """Parse TDCC (集保) data files → per-stock periodic DataFrame.
    
    FinMind format: daily foreign investment data.
    Returns: DataFrame with columns:
      [date, stock_code, foreign_holding_ratio, shares_issued]
    """
    tdcc_dir = DATA_ROOT / 'tdcc'
    rows = []
    
    for f in sorted(tdcc_dir.glob('finmind_*.json')):
        with open(f, 'r', encoding='utf-8') as fp:
            d = json.load(fp)
        
        stock_id = d.get('stock_id', '')
        if not re.match(r'^\d{4}$', stock_id):
            continue
        
        for item in d.get('data', []):
            rows.append({
                'date': pd.Timestamp(item['date']),
                'stock_code': stock_id,
                'foreign_holding_ratio': item.get('ForeignInvestmentSharesRatio', 0) / 100.0,
                'shares_issued': item.get('NumberOfSharesIssued', 0)
            })
    
    df = pd.DataFrame(rows)
    print(f'TDCC data: {len(df)} rows, {df["stock_code"].nunique()} stocks')
    return df

tdcc_df = load_tdcc_data()
tdcc_df.head()

In [ ]:
def load_broker_data() -> pd.DataFrame:
    """Parse broker monthly files → per-stock monthly DataFrame.
    
    Returns: DataFrame with columns:
      [year_month, stock_code, broker_hhi, broker_net_buy_ratio, 
       broker_consistency, broker_spread]
    """
    broker_dir = DATA_ROOT / 'broker'
    rows = []
    
    for f in sorted(broker_dir.glob('*.json')):
        fname = f.stem  # e.g. 1101_202102
        parts = fname.split('_')
        if len(parts) != 2:
            continue
        code, ym = parts
        if not re.match(r'^\d{4}$', code):
            continue
        
        year = int(ym[:4])
        month = int(ym[4:6])
        # Use last day of month as date
        if month == 12:
            date = pd.Timestamp(f'{year}-{month}-31')
        else:
            date = pd.Timestamp(f'{year}-{month+1}-01') - pd.Timedelta(days=1)
        
        with open(f, 'r', encoding='utf-8') as fp:
            d = json.load(fp)
        
        buy_top = d.get('buy_top', [])
        sell_top = d.get('sell_top', [])
        
        # Filter out summary rows
        buys = [b for b in buy_top if '合計' not in b.get('broker', '')]
        sells = [s for s in sell_top if '合計' not in s.get('broker', '')]
        
        # HHI: sum of squared market shares of top 15
        buy_pcts = [parse_number(b.get('pct', '0').replace('%', '')) / 100 for b in buys[:15]]
        sell_pcts = [parse_number(s.get('pct', '0').replace('%', '')) / 100 for s in sells[:15]]
        all_pcts = buy_pcts + sell_pcts
        hhi = sum(p**2 for p in all_pcts) if all_pcts else 0
        
        # Net buy ratio of top 5
        top5_net = sum(parse_number(b.get('net', '0')) for b in buys[:5])
        top5_sell = sum(parse_number(s.get('net', '0')) for s in sells[:5])
        total_action = abs(top5_net) + abs(top5_sell)
        net_buy_ratio = top5_net / total_action if total_action > 0 else 0
        
        # Broker spread: buy broker count vs sell broker count
        broker_spread = len(buys) / max(len(sells), 1)
        
        rows.append({
            'date': date, 'stock_code': code,
            'broker_hhi': hhi,
            'broker_net_buy_ratio': net_buy_ratio,
            'broker_consistency': 0,  # Requires time series — computed later
            'broker_spread': broker_spread
        })
    
    df = pd.DataFrame(rows)
    
    # Compute broker_consistency: consecutive months of net buying (signed)
    df = df.sort_values(['stock_code', 'date'])
    for code, group in df.groupby('stock_code'):
        streak = 0
        for idx in group.index:
            nbr = df.loc[idx, 'broker_net_buy_ratio']
            if nbr > 0:
                streak = max(streak, 0) + 1
            elif nbr < 0:
                streak = min(streak, 0) - 1
            else:
                streak = 0
            df.loc[idx, 'broker_consistency'] = streak
    
    print(f'Broker data: {len(df)} rows, {df["stock_code"].nunique()} stocks')
    return df

broker_df = load_broker_data()
broker_df.head()

In [ ]:
def compute_institutional_features(prices: pd.DataFrame, inst: pd.DataFrame, 
                                    margin: pd.DataFrame, tdcc: pd.DataFrame,
                                    broker: pd.DataFrame) -> pd.DataFrame:
    """Merge institutional sources and compute 15 features.
    
    Features 21-35 from the plan.
    """
    # Start from prices (date, stock_code, volume)
    df = prices[['date', 'stock_code', 'volume']].copy()
    df = df.sort_values(['stock_code', 'date'])
    
    # Merge institutional
    inst = inst.sort_values(['stock_code', 'date'])
    df = df.merge(inst, on=['date', 'stock_code'], how='left')
    
    # Normalize by volume (shares)
    vol = df['volume'].replace(0, np.nan)
    df['inst_foreign_net'] = df['foreign_net'] / vol
    df['inst_trust_net'] = df['trust_net'] / vol
    df['inst_dealer_net'] = df['dealer_net'] / vol
    df['inst_total_net'] = df['total_net'] / vol
    
    # 5-day rolling sum of total institutional net
    df['inst_5d_sum'] = df.groupby('stock_code')['inst_total_net'].transform(
        lambda x: x.rolling(5, min_periods=1).sum()
    )
    
    # Merge margin data
    margin = margin.sort_values(['stock_code', 'date'])
    df = df.merge(margin[['date', 'stock_code', 'margin_balance', 'short_balance', 'margin_utilization']], 
                  on=['date', 'stock_code'], how='left')
    
    # Margin/short balance day-over-day change %
    df['margin_balance_chg'] = df.groupby('stock_code')['margin_balance'].transform(
        lambda x: x.pct_change()
    )
    df['short_balance_chg'] = df.groupby('stock_code')['short_balance'].transform(
        lambda x: x.pct_change()
    )
    # margin_utilization already 0-1
    
    # Merge TDCC (forward-fill since it's not daily for all stocks)
    tdcc = tdcc.sort_values(['stock_code', 'date'])
    df = df.merge(tdcc[['date', 'stock_code', 'foreign_holding_ratio']], 
                  on=['date', 'stock_code'], how='left')
    # Forward-fill TDCC data within each stock
    df['foreign_holding_ratio'] = df.groupby('stock_code')['foreign_holding_ratio'].transform(
        lambda x: x.ffill()
    )
    
    # TDCC features: use foreign_holding_ratio change as proxy for retail/big holder change
    df['tdcc_retail_chg'] = -df.groupby('stock_code')['foreign_holding_ratio'].transform(
        lambda x: x.diff()  # Inverse: when foreign increases, retail decreases
    )
    df['tdcc_big_chg'] = df.groupby('stock_code')['foreign_holding_ratio'].transform(
        lambda x: x.diff()
    )
    df['tdcc_concentration'] = df['foreign_holding_ratio']  # Direct
    
    # Merge broker (monthly → forward-fill to daily)
    broker = broker.sort_values(['stock_code', 'date'])
    df = df.merge(broker[['date', 'stock_code', 'broker_hhi', 'broker_net_buy_ratio', 
                          'broker_consistency', 'broker_spread']], 
                  on=['date', 'stock_code'], how='left')
    # Forward-fill broker data
    for col in ['broker_hhi', 'broker_net_buy_ratio', 'broker_consistency', 'broker_spread']:
        df[col] = df.groupby('stock_code')[col].transform(lambda x: x.ffill())
    
    # Drop intermediate columns
    df = df.drop(columns=['volume', 'foreign_net', 'trust_net', 'dealer_net', 'total_net',
                          'margin_balance', 'short_balance', 'foreign_holding_ratio'], errors='ignore')
    
    return df

inst_features = compute_institutional_features(prices_df, inst_df, margin_df, tdcc_df, broker_df)
inst_cols = ['inst_foreign_net', 'inst_trust_net', 'inst_dealer_net', 'inst_total_net',
             'inst_5d_sum', 'margin_balance_chg', 'short_balance_chg', 'margin_utilization',
             'tdcc_retail_chg', 'tdcc_big_chg', 'tdcc_concentration',
             'broker_hhi', 'broker_net_buy_ratio', 'broker_consistency', 'broker_spread']
print(f'Institutional features: {len(inst_cols)}')
print(inst_features[inst_cols].describe())

## 5. Industry Features (5)

In [ ]:
def load_industry_mapping() -> dict:
    """Load industry chain mapping → {stock_code: {chain_code, chain_name, position}}."""
    industry_dir = DATA_ROOT / 'industry'
    mapping = {}  # stock_code → chain info
    chains = {}   # chain_code → list of stock codes
    
    for f in sorted(industry_dir.glob('ic_chain_*.json')):
        with open(f, 'r', encoding='utf-8') as fp:
            d = json.load(fp)
        chain_code = d.get('chain_code', '')
        chain_name = d.get('chain_name', '')
        stock_codes = d.get('stock_codes', [])
        chains[chain_code] = stock_codes
        
        for i, code in enumerate(stock_codes):
            # Position: normalized index in chain (0=upstream, 1=downstream)
            pos = i / max(len(stock_codes) - 1, 1)
            mapping[code] = {
                'chain_code': chain_code,
                'chain_name': chain_name,
                'industry_chain_pos': pos
            }
    
    print(f'Industry mapping: {len(mapping)} stocks in {len(chains)} chains')
    return mapping, chains


def compute_industry_features(prices: pd.DataFrame, industry_mapping: dict, 
                               chains: dict) -> pd.DataFrame:
    """Compute 5 industry features.
    
    Features 36-40: sector_rs, peer_alpha, sector_momentum, 
                    industry_chain_pos, sector_concentration
    """
    df = prices[['date', 'stock_code', 'close']].copy()
    df = df.sort_values(['stock_code', 'date'])
    
    # Compute per-stock 20d return for RS computation
    df['ret_20d_raw'] = df.groupby('stock_code')['close'].transform(lambda x: x.pct_change(20))
    
    # Map each stock to its chain
    df['chain_code'] = df['stock_code'].map(lambda c: industry_mapping.get(c, {}).get('chain_code', 'UNKNOWN'))
    df['industry_chain_pos'] = df['stock_code'].map(
        lambda c: industry_mapping.get(c, {}).get('industry_chain_pos', 0.5)  # Default middle
    )
    
    # Sector RS: median 20d return per chain per date
    sector_rs = df.groupby(['date', 'chain_code'])['ret_20d_raw'].median().reset_index()
    sector_rs.columns = ['date', 'chain_code', 'sector_momentum']
    df = df.merge(sector_rs, on=['date', 'chain_code'], how='left')
    
    # Sector RS rating: rank sectors by momentum, percentile
    sector_rank = df.groupby('date')['sector_momentum'].rank(pct=True)
    df['sector_rs'] = sector_rank
    
    # Peer Alpha: stock return / sector return
    df['peer_alpha'] = df['ret_20d_raw'] / df['sector_momentum'].replace(0, np.nan)
    df['peer_alpha'] = df['peer_alpha'].clip(-5, 5)  # Cap outliers
    
    # Sector concentration: count stocks in same sector on same date / total stocks
    chain_counts = df.groupby(['date', 'chain_code'])['stock_code'].transform('count')
    total_counts = df.groupby('date')['stock_code'].transform('count')
    df['sector_concentration'] = chain_counts / total_counts
    
    result = df[['date', 'stock_code', 'sector_rs', 'peer_alpha', 'sector_momentum',
                 'industry_chain_pos', 'sector_concentration']].copy()
    
    print(f'Industry features computed')
    return result

industry_mapping, chains = load_industry_mapping()
industry_features = compute_industry_features(prices_df, industry_mapping, chains)
print(industry_features.describe())

## 6. Fundamental Features (8) — point-in-time forward-fill

In [ ]:
def load_financials_data() -> pd.DataFrame:
    """Parse financials (balance sheet) files → per-stock quarterly DataFrame.
    
    Returns: DataFrame with columns:
      [date, stock_code, type, value]
    """
    fin_dir = DATA_ROOT / 'financials'
    all_rows = []
    
    for f in sorted(fin_dir.glob('balance_sheet_*.json')):
        with open(f, 'r', encoding='utf-8') as fp:
            d = json.load(fp)
        
        stock_id = d.get('stock_id', '')
        if not re.match(r'^\d{4}$', stock_id):
            continue
        
        for item in d.get('data', []):
            all_rows.append({
                'date': pd.Timestamp(item['date']),
                'stock_code': stock_id,
                'type': item.get('type', ''),
                'value': item.get('value', 0)
            })
    
    df = pd.DataFrame(all_rows)
    print(f'Financials raw: {len(df)} rows, {df["stock_code"].nunique()} stocks')
    return df


def load_revenue_data() -> pd.DataFrame:
    """Parse revenue monthly files → per-stock monthly DataFrame."""
    rev_dir = DATA_ROOT / 'revenue'
    rows = []
    
    for f in sorted(rev_dir.glob('otc_*.json')):
        with open(f, 'r', encoding='utf-8') as fp:
            d = json.load(fp)
        
        year_minguo = d.get('year_minguo', 0)
        month = d.get('month', 0)
        year = year_minguo + 1911
        if month == 12:
            date = pd.Timestamp(f'{year}-{month}-31')
        else:
            date = pd.Timestamp(f'{year}-{month+1}-01') - pd.Timedelta(days=1)
        
        for item in d.get('data', []):
            code = item.get('code', '').strip()
            if not re.match(r'^\d{4}$', code):
                continue
            rows.append({
                'date': date,
                'stock_code': code,
                'revenue': parse_number(item.get('revenue', '0')),
                'yoy_pct': parse_number(item.get('yoy_pct', '0')),
                'mom_pct': parse_number(item.get('mom_pct', '0')),
            })
    
    df = pd.DataFrame(rows)
    print(f'Revenue data: {len(df)} rows, {df["stock_code"].nunique()} stocks')
    return df


def compute_fundamental_features(prices: pd.DataFrame, financials_raw: pd.DataFrame,
                                  revenue: pd.DataFrame) -> pd.DataFrame:
    """Compute 8 fundamental features with point-in-time forward-fill.
    
    Features 41-48: eps_yoy, roe, revenue_yoy, revenue_mom, 
                    pe_percentile, pb_ratio, operating_margin, debt_ratio
    """
    # Pivot financials by type
    useful_types = {
        'ReturnOnEquity': 'roe',
        'EarningsPerShare': 'eps',
        'PriceEarningRatio': 'pe_ratio',
        'PriceBookRatio': 'pb_ratio',
        'OperatingProfitMargin': 'operating_margin',
        'OperatingProfitMargin_per': 'operating_margin',
        'DebtRatio': 'debt_ratio',
        'DebtRatio_per': 'debt_ratio',
    }
    
    # Filter to useful types
    fin = financials_raw[financials_raw['type'].isin(useful_types.keys())].copy()
    fin['feature'] = fin['type'].map(useful_types)
    
    # Pivot: one row per (stock, date), columns are features
    fin_pivot = fin.pivot_table(index=['stock_code', 'date'], columns='feature', 
                                values='value', aggfunc='first').reset_index()
    fin_pivot = fin_pivot.sort_values(['stock_code', 'date'])
    
    # EPS YoY: compare to same quarter last year
    if 'eps' in fin_pivot.columns:
        fin_pivot['eps_yoy'] = fin_pivot.groupby('stock_code')['eps'].transform(
            lambda x: x.pct_change(4)  # 4 quarters = 1 year
        )
    else:
        fin_pivot['eps_yoy'] = np.nan
    
    # Ensure all columns exist
    for col in ['roe', 'pe_ratio', 'pb_ratio', 'operating_margin', 'debt_ratio']:
        if col not in fin_pivot.columns:
            fin_pivot[col] = np.nan
    
    # Normalize: ROE / operating_margin as ratios; debt_ratio already 0-100
    fin_pivot['roe'] = fin_pivot['roe'] / 100.0
    fin_pivot['operating_margin'] = fin_pivot['operating_margin'] / 100.0
    fin_pivot['debt_ratio'] = fin_pivot['debt_ratio'] / 100.0
    
    # PE Percentile: rank within sector (simplified: rank across all stocks)
    fin_pivot['pe_percentile'] = fin_pivot.groupby('date')['pe_ratio'].rank(pct=True)
    
    # Revenue features
    rev = revenue.copy()
    rev['revenue_yoy'] = rev['yoy_pct'] / 100.0
    rev['revenue_mom'] = rev['mom_pct'] / 100.0
    
    # Merge fundamentals to daily prices via forward-fill
    base = prices[['date', 'stock_code']].copy()
    base = base.sort_values(['stock_code', 'date'])
    
    # Merge financials (quarterly → forward-fill)
    fin_cols = ['date', 'stock_code', 'eps_yoy', 'roe', 'pe_percentile', 
                'pb_ratio', 'operating_margin', 'debt_ratio']
    fin_select = fin_pivot[[c for c in fin_cols if c in fin_pivot.columns]].copy()
    base = base.merge(fin_select, on=['date', 'stock_code'], how='left')
    
    # Merge revenue (monthly → forward-fill)
    rev_cols = ['date', 'stock_code', 'revenue_yoy', 'revenue_mom']
    base = base.merge(rev[rev_cols], on=['date', 'stock_code'], how='left')
    
    # Forward-fill all fundamental features
    fund_feature_cols = ['eps_yoy', 'roe', 'revenue_yoy', 'revenue_mom',
                         'pe_percentile', 'pb_ratio', 'operating_margin', 'debt_ratio']
    for col in fund_feature_cols:
        if col not in base.columns:
            base[col] = np.nan
        base[col] = base.groupby('stock_code')[col].transform(lambda x: x.ffill())
    
    print(f'Fundamental features computed')
    return base[['date', 'stock_code'] + fund_feature_cols]

financials_raw = load_financials_data()
revenue_df = load_revenue_data()
fund_features = compute_fundamental_features(prices_df, financials_raw, revenue_df)
print(fund_features.describe())

## 7. News Features (3) — forward-fill, default 0

In [ ]:
def load_news_data() -> pd.DataFrame:
    """Parse news JSON files → per-date article counts.
    
    Note: cnyes news data is aggregated (not per-stock), so we extract
    stock codes mentioned in articles and create per-stock daily counts.
    """
    news_dir = DATA_ROOT / 'news'
    # stock_code → date → count
    mentions = defaultdict(lambda: defaultdict(int))
    
    for f in sorted(news_dir.glob('cnyes_*.json')):
        with open(f, 'r', encoding='utf-8') as fp:
            d = json.load(fp)
        
        fname = f.stem  # cnyes_20260207_20260207_p0023
        parts = fname.split('_')
        if len(parts) < 3:
            continue
        date_str = parts[1]
        date = pd.Timestamp(f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}')
        
        for article in d.get('data', []):
            content = str(article.get('content', ''))
            # Extract 4-digit stock codes from content
            found_codes = re.findall(r'(?:TWS:|TWG:)(\d{4})', content)
            for code in set(found_codes):
                mentions[code][date] += 1
    
    rows = []
    for code, date_counts in mentions.items():
        for date, count in date_counts.items():
            rows.append({'date': date, 'stock_code': code, 'news_count': count})
    
    df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=['date', 'stock_code', 'news_count'])
    print(f'News data: {len(df)} rows, {df["stock_code"].nunique()} stocks')
    return df


def compute_news_features(prices: pd.DataFrame, news: pd.DataFrame) -> pd.DataFrame:
    """Compute 3 news features.
    
    Features 49-51: news_count_7d, news_sentiment, news_volume_spike
    """
    base = prices[['date', 'stock_code']].copy()
    base = base.sort_values(['stock_code', 'date'])
    
    if len(news) > 0:
        base = base.merge(news[['date', 'stock_code', 'news_count']], 
                          on=['date', 'stock_code'], how='left')
    else:
        base['news_count'] = 0
    
    base['news_count'] = base['news_count'].fillna(0)
    
    # 7-day rolling news count
    base['news_count_7d'] = base.groupby('stock_code')['news_count'].transform(
        lambda x: x.rolling(7, min_periods=1).sum()
    )
    
    # News sentiment: placeholder (cnyes data doesn't have explicit sentiment)
    # Default to 0 (neutral)
    base['news_sentiment'] = 0.0
    
    # News volume spike: news_count / 30d avg
    avg_30d = base.groupby('stock_code')['news_count'].transform(
        lambda x: x.rolling(30, min_periods=1).mean()
    )
    base['news_volume_spike'] = base['news_count'] / avg_30d.replace(0, np.nan)
    base['news_volume_spike'] = base['news_volume_spike'].fillna(0)
    
    return base[['date', 'stock_code', 'news_count_7d', 'news_sentiment', 'news_volume_spike']]

news_df = load_news_data()
news_features = compute_news_features(prices_df, news_df)
print(news_features.describe())

## 8. RS Rating vs TAIEX

Compute relative strength of each stock vs TAIEX index.

In [ ]:
def compute_rs_rating(prices: pd.DataFrame) -> pd.DataFrame:
    """Compute RS Rating = stock 63d return / TAIEX 63d return, ranked as percentile."""
    # Fetch TAIEX
    taiex = yf.download('^TWII', start=START_DATE, auto_adjust=True, progress=False)
    taiex.columns = [c.lower() if isinstance(c, str) else c[0].lower() for c in taiex.columns]
    taiex = taiex[['close']].rename(columns={'close': 'taiex_close'})
    taiex.index.name = 'date'
    taiex = taiex.reset_index()
    taiex['date'] = pd.to_datetime(taiex['date'])
    taiex['taiex_ret_63'] = taiex['taiex_close'].pct_change(63)
    
    # Stock 63d returns
    df = prices[['date', 'stock_code', 'close']].copy()
    df = df.sort_values(['stock_code', 'date'])
    df['stock_ret_63'] = df.groupby('stock_code')['close'].transform(lambda x: x.pct_change(63))
    
    # Merge TAIEX
    df = df.merge(taiex[['date', 'taiex_ret_63']], on='date', how='left')
    
    # RS = stock_ret / taiex_ret (weighted: base^0.6 * recent^0.4)
    df['rs_raw'] = df['stock_ret_63'] - df['taiex_ret_63']
    
    # Rank RS as percentile within each date
    df['rs_rating'] = df.groupby('date')['rs_raw'].rank(pct=True)
    
    return df[['date', 'stock_code', 'rs_rating']]

rs_df = compute_rs_rating(prices_df)
print(f'RS Rating computed: {rs_df.shape}')
print(rs_df.describe())

## 9. Merge All Features + Z-Score Normalize

In [ ]:
# === Step 1: Compute technical features for all stocks ===
print('Computing technical features for all stocks...')
tech_results = []
for code, group in prices_df.groupby('stock_code'):
    tech = compute_technical_features(group)
    tech_results.append(tech)

tech_all = pd.concat(tech_results, ignore_index=True)
print(f'Technical features: {tech_all.shape}')

# Update rs_rating from dedicated computation
tech_all = tech_all.drop(columns=['rs_rating'], errors='ignore')
tech_all = tech_all.merge(rs_df, on=['date', 'stock_code'], how='left')

In [ ]:
# === Step 2: Merge all feature groups ===
print('Merging all feature groups...')

# Start with technical (has date, stock_code, + OHLCV + 20 tech features)
merged = tech_all[['date', 'stock_code'] + tech_cols].copy()

# Merge institutional features
merged = merged.merge(
    inst_features[['date', 'stock_code'] + inst_cols],
    on=['date', 'stock_code'], how='left'
)

# Merge industry features
industry_cols = ['sector_rs', 'peer_alpha', 'sector_momentum', 'industry_chain_pos', 'sector_concentration']
merged = merged.merge(
    industry_features[['date', 'stock_code'] + industry_cols],
    on=['date', 'stock_code'], how='left'
)

# Merge fundamental features
fund_cols = ['eps_yoy', 'roe', 'revenue_yoy', 'revenue_mom', 
             'pe_percentile', 'pb_ratio', 'operating_margin', 'debt_ratio']
merged = merged.merge(
    fund_features[['date', 'stock_code'] + fund_cols],
    on=['date', 'stock_code'], how='left'
)

# Merge news features
news_cols = ['news_count_7d', 'news_sentiment', 'news_volume_spike']
merged = merged.merge(
    news_features[['date', 'stock_code'] + news_cols],
    on=['date', 'stock_code'], how='left'
)

# All 51 feature columns
ALL_FEATURE_COLS = tech_cols + inst_cols + industry_cols + fund_cols + news_cols
assert len(ALL_FEATURE_COLS) == 51, f'Expected 51 features, got {len(ALL_FEATURE_COLS)}'

print(f'Merged shape: {merged.shape}')
print(f'Feature columns: {len(ALL_FEATURE_COLS)}')
print(f'Stocks: {merged["stock_code"].nunique()}')
print(f'Date range: {merged["date"].min()} to {merged["date"].max()}')
print(f'\nMissing values per feature:')
print(merged[ALL_FEATURE_COLS].isnull().mean().sort_values(ascending=False).head(10))

In [ ]:
# === Step 3: Z-Score Normalize (rolling 252d window) ===
print('Z-score normalizing features (rolling 252d)...')

merged = merged.sort_values(['stock_code', 'date'])

for col in ALL_FEATURE_COLS:
    # Replace inf with nan before normalization
    merged[col] = merged[col].replace([np.inf, -np.inf], np.nan)
    
    # Rolling Z-score per stock
    grouped = merged.groupby('stock_code')[col]
    rolling_mean = grouped.transform(lambda x: x.rolling(252, min_periods=60).mean())
    rolling_std = grouped.transform(lambda x: x.rolling(252, min_periods=60).std())
    merged[col] = (merged[col] - rolling_mean) / rolling_std.replace(0, np.nan)
    
    # Clip extreme Z-scores
    merged[col] = merged[col].clip(-5, 5)

# Fill remaining NaN with 0 (no signal)
merged[ALL_FEATURE_COLS] = merged[ALL_FEATURE_COLS].fillna(0)

print(f'Normalized shape: {merged.shape}')
print(merged[ALL_FEATURE_COLS].describe())

## 10. Forward Returns

In [ ]:
# === Compute forward returns: D3, D7, D21, D90, D180 ===
print('Computing forward returns...')

# Need original prices for forward returns
price_lookup = prices_df[['date', 'stock_code', 'close']].sort_values(['stock_code', 'date'])

horizons = {'d3': 3, 'd7': 7, 'd21': 21, 'd90': 90, 'd180': 180}

for name, days in horizons.items():
    price_lookup[name] = price_lookup.groupby('stock_code')['close'].transform(
        lambda x: x.shift(-days) / x - 1
    )

fwd_returns = price_lookup[['date', 'stock_code'] + list(horizons.keys())].copy()
print(f'Forward returns shape: {fwd_returns.shape}')
print(fwd_returns[list(horizons.keys())].describe())

## 11. Save Outputs

In [ ]:
# === Save features_all.parquet ===
output_cols = ['date', 'stock_code'] + ALL_FEATURE_COLS
features_out = merged[output_cols].copy()
features_out.to_parquet(OUTPUT_DIR / 'features_all.parquet', index=False)
print(f'Saved features_all.parquet: {features_out.shape}')

# === Save forward_returns.parquet ===
fwd_returns.to_parquet(OUTPUT_DIR / 'forward_returns.parquet', index=False)
print(f'Saved forward_returns.parquet: {fwd_returns.shape}')

# === Save feature_metadata.json ===
metadata = {
    'dimensions': {
        'technical': {
            'features': tech_cols,
            'count': len(tech_cols),
            'description': '技術面 — 價量衍生指標'
        },
        'institutional': {
            'features': inst_cols,
            'count': len(inst_cols),
            'description': '籌碼面 — 法人/融資融券/集保/主力'
        },
        'industry': {
            'features': industry_cols,
            'count': len(industry_cols),
            'description': '產業面 — 族群強弱/供應鏈'
        },
        'fundamental': {
            'features': fund_cols,
            'count': len(fund_cols),
            'description': '基本面 — 財報/營收/估值'
        },
        'news': {
            'features': news_cols,
            'count': len(news_cols),
            'description': '新聞面 — 媒體關注度'
        }
    },
    'all_features': ALL_FEATURE_COLS,
    'total_features': len(ALL_FEATURE_COLS),
    'forward_return_horizons': list(horizons.keys()),
    'normalization': 'rolling_zscore_252d',
    'created_at': datetime.now().isoformat(),
    'stock_count': int(features_out['stock_code'].nunique()),
    'date_range': [str(features_out['date'].min()), str(features_out['date'].max())]
}

with open(OUTPUT_DIR / 'feature_metadata.json', 'w', encoding='utf-8') as fp:
    json.dump(metadata, fp, indent=2, ensure_ascii=False)

print(f'\n=== Summary ===')
print(f'Features: {len(ALL_FEATURE_COLS)} across {len(metadata["dimensions"])} dimensions')
for dim, info in metadata['dimensions'].items():
    print(f'  {dim}: {info["count"]} features — {info["description"]}')
print(f'Stocks: {metadata["stock_count"]}')
print(f'Date range: {metadata["date_range"]}')
print(f'\nOutput files:')
for f in OUTPUT_DIR.glob('*'):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name}: {size_mb:.1f} MB')